# 03.1 - Đặc trưng thô theo cụm để tự gán nhãn


Notebook phụ này chỉ đọc kết quả đã có `cluster_id` từ `results/clean_data_train_clustered.csv`. Mục tiêu là liệt kê đặc trưng nổi bật của từng cụm trước khi đặt `cluster_label`, để người phân tích tự suy luận tên nhãn dựa trên bằng chứng dữ liệu.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_columns', 50)

DATA_PATH = Path('../results/clean_data_train_clustered.csv')
assert DATA_PATH.exists(), f'Không tìm thấy file: {DATA_PATH}'

## 1. Nạp dữ liệu đã có `cluster_id`

In [2]:
usecols = [
    'cluster_id',
    'job_title', 'job_industry', 'location', 'job_type', 'education_level',
    'salary_min_m_vnd', 'salary_max_m_vnd', 'exp_min_years', 'exp_max_years',
    'text_combined'
]

df = pd.read_csv(DATA_PATH, usecols=usecols)
df['cluster_id'] = df['cluster_id'].astype(int)

print(f'Số dòng train đã phân cụm: {len(df):,}')
print(f'Số cụm: {df["cluster_id"].nunique()}')
display(df[['cluster_id', 'job_title', 'job_industry', 'salary_min_m_vnd', 'exp_min_years']].head())

Số dòng train đã phân cụm: 523,972
Số cụm: 17


,cluster_id,job_title,job_industry,salary_min_m_vnd,exp_min_years
0,7,nhân viên kinh doanh thu nhập đến 30 triệu đi làm ngay,Xây dựng,12.0,6.0
1,7,kỹ sư giám sát xây dựng,Xây dựng,10.0,5.0
2,4,tuyển sales telesales cho công ty bhnt prudential,Chưa xác định,5.0,1.0
3,7,kế toán tổng hợp,Kế toán / Kiểm toán,8.0,5.0
4,7,tuyển dụng nhân viêv video editor,Marketing,8.0,2.0


## 2. Hàm thống kê đặc trưng cụm


Các hàm dưới đây lấy top ngành, top chức danh, top học vấn, lương/kinh nghiệm trung vị và từ khóa TF-IDF nổi bật trong từng cụm.

In [3]:
def top_with_pct(s, n=3):
    vc = s.fillna('Không xác định').value_counts(dropna=False).head(n)
    total = len(s)
    return ', '.join([f'{idx} ({cnt / total * 100:.1f}%)' for idx, cnt in vc.items()])


def clean_for_tfidf(text):
    text = '' if pd.isna(text) else str(text).lower()
    text = re.sub(r'[^0-9a-zA-ZÀ-ỹ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


VI_STOPWORDS = {
    'và', 'của', 'các', 'cho', 'trong', 'với', 'được', 'theo', 'tại', 'làm', 'việc',
    'có', 'là', 'ứng', 'viên', 'nhân', 'sự', 'công', 'ty', 'mô', 'tả', 'yêu', 'cầu',
    'quyền', 'lợi', 'kinh', 'nghiệm', 'năm', 'tháng', 'ngày', 'khi', 'sẽ', 'đến',
    'từ', 'hoặc', 'một', 'khác', 'liên', 'quan', 'ưu', 'tiên', 'không', 'trình', 'độ',
    'khả', 'năng', 'tốt', 'cao', 'thực', 'hiện', 'hỗ', 'trợ', 'đảm', 'bảo'
}


def top_tfidf_terms(texts, n=8):
    texts = [clean_for_tfidf(t) for t in texts if isinstance(t, str) and len(t) > 20]
    if len(texts) < 5:
        return 'Không đủ văn bản'
    vectorizer = TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.85,
        stop_words=list(VI_STOPWORDS)
    )
    X = vectorizer.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = np.array(vectorizer.get_feature_names_out())
    top_idx = scores.argsort()[::-1][:n]
    return ', '.join(terms[top_idx])

## 3. Bảng đặc trưng của từng cụm

In [4]:
SAMPLE_PER_CLUSTER_FOR_KEYWORDS = 3000
rng = np.random.default_rng(42)

rows = []
for cid, g in df.groupby('cluster_id', sort=True):
    keyword_sample = g['text_combined']
    if len(keyword_sample) > SAMPLE_PER_CLUSTER_FOR_KEYWORDS:
        keyword_sample = keyword_sample.sample(SAMPLE_PER_CLUSTER_FOR_KEYWORDS, random_state=42)
    
    rows.append({
        'cluster_id': cid,
        'số tin': len(g),
        'tỷ lệ (%)': len(g) / len(df) * 100,
        'lương min/max trung vị': f"{g['salary_min_m_vnd'].median():.1f} - {g['salary_max_m_vnd'].median():.1f} triệu",
        'KN min/max trung vị': f"{g['exp_min_years'].median():.1f} - {g['exp_max_years'].median():.1f} năm",
        'top ngành': top_with_pct(g['job_industry'], 4),
        'top chức danh': top_with_pct(g['job_title'], 4),
        'top học vấn': top_with_pct(g['education_level'], 3),
        'top hình thức': top_with_pct(g['job_type'], 3),
        'top địa điểm': top_with_pct(g['location'], 3),
        'từ khóa nổi bật': top_tfidf_terms(keyword_sample, 10),
    })

raw_cluster_evidence_df = pd.DataFrame(rows)
display(raw_cluster_evidence_df)

,cluster_id,số tin,tỷ lệ (%),lương min/max trung vị,KN min/max trung vị,top ngành,top chức danh,top học vấn,top hình thức,top địa điểm,từ khóa nổi bật
0,0,9771,1.864794,9.0 - 15.0 triệu,3.0 - 3.0 năm,Thực phẩm - Đồ uống / Công nghệ thực phẩm - Dinh dưỡng (100.0%),"nhân viên kinh doanh (2.2%), nhân viên bán hàng (1.3%), nhân viên qc (0.9%), nhân viên kinh doanh thị trường (0.6%)","Không (46.8%), Cao đẳng (19.5%), Trung cấp (13.6%)","Toàn thời gian (92.0%), Khác (3.0%), Thực tập (2.8%)","Hồ Chí Minh (66.5%), Hà Nội (12.8%), Bình Dương (5.9%)","khách, sản, khách hàng, phẩm, lý, quản, quản lý, bán, doanh, động"
1,1,50957,9.725138,9.0 - 15.0 triệu,2.0 - 2.0 năm,"Bán hàng - Kinh doanh (99.9%), Bán sỉ - Bán lẻ - Quản lý cửa hàng (0.0%), Khách sạn - Nhà hàng - Du lịch / Bán sỉ - Bán lẻ - Quản lý cửa hàng (0.0%), Xây dự...","nhân viên kinh doanh (13.0%), nhân viên bán hàng (2.4%), nhân viên kinh doanh thị trường (1.7%), trưởng phòng kinh doanh (1.0%)","Không (35.2%), Cao đẳng (24.5%), Trung cấp (20.6%)","Toàn thời gian (94.1%), Thực tập (2.6%), Khác (1.5%)","Hồ Chí Minh (51.5%), Hà Nội (27.4%), Bình Dương (3.3%)","khách hàng, doanh, bán, động, lý, bán hàng, lương, sản, phẩm, thưởng"
2,2,13080,2.496317,9.0 - 15.0 triệu,3.0 - 3.0 năm,"Dược phẩm / Y tế - Chăm sóc sức khỏe (98.3%), Y tế - Chăm sóc sức khỏe (0.4%), Chăm sóc khách hàng (0.3%), Chăm sóc sức khỏe / Y tế (0.1%)","nhân viên kinh doanh (2.4%), trình dược viên (1.3%), trình dược viên otc (0.9%), điều dưỡng viên (0.9%)","Trung cấp (31.8%), Không (29.5%), Cao đẳng (20.8%)","Toàn thời gian (97.6%), Khác (0.9%), Thực tập (0.8%)","Hồ Chí Minh (57.7%), Hà Nội (21.0%), Long An (2.5%)","hàng, khách, khách hàng, phẩm, doanh, định, sản, bán, dược, sản phẩm"
3,3,14804,2.825342,9.0 - 15.0 triệu,1.0 - 1.0 năm,"IT phần mềm (11.5%), Nhân viên kinh doanh (10.5%), Marketing - PR (6.4%), Tư vấn/ Chăm sóc khách hàng (5.3%)","kế toán tổng hợp (0.5%), kỹ sư tự động hóa (0.4%), kiến trúc sư (0.3%), devops engineer (0.3%)","Không (52.6%), Đại học (25.6%), Cao đẳng (13.1%)","Toàn thời gian (84.7%), Thực tập (4.0%), Không (3.9%)","Hồ Chí Minh (46.5%), Hà Nội (31.7%), Đà Nẵng (2.6%)","and, hàng, to, in, the, khách, lý, kế, khách hàng, kỹ"
4,4,21895,4.178658,9.0 - 15.0 triệu,3.0 - 3.0 năm,Chưa xác định (100.0%),"nhân viên kinh doanh (3.6%), nhân viên bảo vệ (1.1%), nhân viên marketing (0.9%), nhân viên content marketing (0.6%)","Không (40.6%), Cao đẳng (23.6%), Trung cấp (15.7%)","Toàn thời gian (95.6%), Thực tập (1.7%), Khác (1.4%)","Hồ Chí Minh (49.1%), Hà Nội (28.0%), Bình Dương (3.8%)","hàng, khách, khách hàng, sản, doanh, động, nghiệp, lý, lương, định"
5,5,13219,2.522845,9.0 - 15.0 triệu,3.0 - 3.0 năm,"Giáo dục - Đào tạo / Chăm sóc khách hàng (92.8%), Giáo dục - Đào tạo / Biên phiên dịch (4.2%), Thông tin - Truyền thông - Quảng cáo (1.0%), Giáo dục - Đào t...","nhân viên tư vấn tuyển sinh (2.3%), giáo viên mầm non (2.0%), giáo viên tiếng anh (2.0%), nhân viên kinh doanh (1.3%)","Cao đẳng (31.3%), Không (30.6%), Đại học (26.3%)","Toàn thời gian (91.1%), Thực tập (5.0%), Khác (1.9%)","Hồ Chí Minh (51.7%), Hà Nội (26.1%), Thanh Hóa (3.2%)","hàng, giáo, khách, khách hàng, dạy, tạo, vấn, sinh, động, trường"
6,6,8278,1.579855,7.0 - 10.0 triệu,2.0 - 2.0 năm,Khách sạn - Nhà hàng - Du lịch (100.0%),"nhân viên phục vụ (4.2%), nhân viên lễ tân (2.3%), quản lý nhà hàng (1.7%), nhân viên kinh doanh (1.4%)","Không (47.0%), Cao đẳng (17.8%), Trung cấp (15.1%)","Toàn thời gian (89.9%), Thực tập (4.5%), Khác (2.7%)","Hồ Chí Minh (55.4%), Hà Nội (23.7%), Đà Nẵng (3.6%)","hàng, khách, vụ, lý, nhà, khách hàng, ca, nhà hàng, quản, lương"
7,7,138937,26.516112,10.0 - 15.0 triệu,3.0 - 3.0 năm,"Kế toán / Kiểm toán (31.7%), Xây dựng (17.8%), Giáo dục - Đào tạo / Biên phiên dịch (10.3%), Marketing (9.0%)","kế toán tổng hợp (4.2%), nhân viên kế toán (2.6%), kế toán viên (1.7%), kế toán nội bộ (1.6%)","Cao đẳng (34.6%), Đại học (27.8%), Không (24.4%)","Toàn thời gian (99.3%), Bán thời gian (0.3%), Khác (0.2%)","Hồ Chí Minh (49.4%)

## 4. In hồ sơ từng cụm để tự đặt nhãn


Mỗi cụm được in ra dưới dạng hồ sơ thô. Dựa vào các bằng chứng này, người phân tích tự điền tên nhãn phù hợp.

In [5]:
for _, row in raw_cluster_evidence_df.iterrows():
    print('=' * 120)
    print(f"CỤM {row['cluster_id']:02d} | TÊN NHÃN TỰ SUY LUẬN: ____________________")
    print(f"- Quy mô: {row['số tin']:,} tin ({row['tỷ lệ (%)']:.2f}%)")
    print(f"- Lương trung vị: {row['lương min/max trung vị']}")
    print(f"- Kinh nghiệm trung vị: {row['KN min/max trung vị']}")
    print(f"- Ngành phổ biến: {row['top ngành']}")
    print(f"- Chức danh phổ biến: {row['top chức danh']}")
    print(f"- Học vấn phổ biến: {row['top học vấn']}")
    print(f"- Hình thức phổ biến: {row['top hình thức']}")
    print(f"- Địa điểm phổ biến: {row['top địa điểm']}")
    print(f"- Từ khóa nổi bật: {row['từ khóa nổi bật']}")

CỤM 00 | TÊN NHÃN TỰ SUY LUẬN: ____________________
- Quy mô: 9,771 tin (1.86%)
- Lương trung vị: 9.0 - 15.0 triệu
- Kinh nghiệm trung vị: 3.0 - 3.0 năm
- Ngành phổ biến: Thực phẩm - Đồ uống / Công nghệ thực phẩm - Dinh dưỡng (100.0%)
- Chức danh phổ biến: nhân viên kinh doanh (2.2%), nhân viên bán hàng (1.3%), nhân viên qc (0.9%), nhân viên kinh doanh thị trường (0.6%)
- Học vấn phổ biến: Không (46.8%), Cao đẳng (19.5%), Trung cấp (13.6%)
- Hình thức phổ biến: Toàn thời gian (92.0%), Khác (3.0%), Thực tập (2.8%)
- Địa điểm phổ biến: Hồ Chí Minh (66.5%), Hà Nội (12.8%), Bình Dương (5.9%)
- Từ khóa nổi bật: khách, sản, khách hàng, phẩm, lý, quản, quản lý, bán, doanh, động
CỤM 01 | TÊN NHÃN TỰ SUY LUẬN: ____________________
- Quy mô: 50,957 tin (9.73%)
- Lương trung vị: 9.0 - 15.0 triệu
- Kinh nghiệm trung vị: 2.0 - 2.0 năm
- Ngành phổ biến: Bán hàng - Kinh doanh (99.9%), Bán sỉ - Bán lẻ - Quản lý cửa hàng (0.0%), Khách sạn - Nhà hàng - Du lịch / Bán sỉ - Bán lẻ - Quản lý cửa hàng (0.0%)

## 5. Lưu bảng ra CSV

In [6]:
OUT_PATH = Path('../outputs/tables/stage_03_cluster_raw_evidence_for_labeling.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
raw_cluster_evidence_df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f'Đã lưu bảng đặc trưng thô để tự gán nhãn cụm tại: {OUT_PATH}')

Đã lưu bảng đặc trưng thô để tự gán nhãn cụm tại: ..\outputs\tables\stage_03_cluster_raw_evidence_for_labeling.csv
